# Prepare Excel Template
This notebook loads an Excel file from `01-INPUT-DATA`, copies it into a template folder, and adds a first sheet for future `config_new.py` inputs.

Fill only the **value** column in the generated `CONFIG_INPUTS` sheet.

In [1]:
from pathlib import Path
import shutil
from datetime import datetime
import openpyxl

# --- User settings ---
# Set this if you want a specific source file; leave as None to auto-pick the first .xlsx file.
SOURCE_EXCEL = None
# Name of the folder where templates are stored (inside 01-INPUT-DATA).
TEMPLATE_FOLDER_NAME = "TEMPLATES"
# Name of the config sheet inserted at the first position.
CONFIG_SHEET_NAME = "CONFIG_INPUTS"

project_root = Path.cwd()
input_data_dir = project_root / "01-INPUT-DATA"
template_dir = input_data_dir / TEMPLATE_FOLDER_NAME
template_dir.mkdir(parents=True, exist_ok=True)

if not input_data_dir.exists():
    raise FileNotFoundError(f"Input data directory not found: {input_data_dir}")

if SOURCE_EXCEL is not None:
    source_excel = Path(SOURCE_EXCEL)
    if not source_excel.is_absolute():
        source_excel = input_data_dir / source_excel
    if not source_excel.exists():
        raise FileNotFoundError(f"Configured SOURCE_EXCEL does not exist: {source_excel}")
else:
    candidates = sorted(
        p for p in input_data_dir.rglob("*.xlsx")
        if TEMPLATE_FOLDER_NAME not in p.parts and not p.name.startswith("~$")
    )
    if not candidates:
        raise FileNotFoundError(
            f"No .xlsx files found under {input_data_dir}. Put one file in 01-INPUT-DATA first."
        )
    source_excel = candidates[0]


In [2]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
template_name = f"template_{source_excel.stem}_{timestamp}.xlsx"
template_path = template_dir / template_name

# 1) Copy source file to template folder
shutil.copy2(source_excel, template_path)

WindowsPath('c:/Users/josua/OneDrive/Dokumente/ETH/Code/CS/CS1-Swiss-Post/01-INPUT-DATA/TEMPLATES/template_20251023_Standortdaten_Vétroz_20260320_150043.xlsx')

In [ ]:
# 2) Add config input sheet as first sheet
wb = openpyxl.load_workbook(template_path)

if CONFIG_SHEET_NAME in wb.sheetnames:
    config_sheet = wb[CONFIG_SHEET_NAME]
    wb.remove(config_sheet)

config_sheet = wb.create_sheet(CONFIG_SHEET_NAME, 0)
config_sheet.append(["parameter", "value", "description", "example"])

config_rows = [
    ["use_case", "", "Optimization use case", "Peak_Shaving"],
    ["interest_rate", "", "Discount/interest rate [-]", "0.06"],
    ["lifetime", "", "Project lifetime [years]", "20"],
    ["year", "", "Planning year", "2025"],
    ["load_existing_input_dict", "", "Load existing input_dict.json [True/False]", "True"],
    ["max_timesteps", "", "Max optimization timesteps (int or None)", "40000"],
    ["optimization_mode", "", "Optimization mode [milp/lp]", "lp"],
    ["PV_max_capacity", "", "Maximum PV capacity [kW]", "10000"],
    ["Battery_max_inflow", "", "Battery max charging power [kW]", "1000"],
    ["Battery_max_outflow", "", "Battery max discharging power [kW]", "1000"],
    ["Battery_max_capacity", "", "Battery max energy capacity [kWh]", "100000"],
    ["eta_charge", "", "Battery charging efficiency [-]", "0.9"],
    ["eta_discharge", "", "Battery discharging efficiency [-]", "0.95"],
    ["eta_self_discharge", "", "Battery self-discharge per step [-]", "0.0"],
    ["invest_cost", "", "Battery investment cost [CHF/kWh]", "450"],
    ["operation_and_maintenance", "", "Annual O&M cost [CHF]; leave empty to auto-calc", "10000"],
    ["battery_degrading", "", "Battery degradation per year [-]", "0.01"],
]

for row in config_rows:
    config_sheet.append(row)

# Keep the sheet easy to fill in manually.
config_sheet.freeze_panes = "A2"
config_sheet.column_dimensions["A"].width = 30
config_sheet.column_dimensions["B"].width = 18
config_sheet.column_dimensions["C"].width = 55
config_sheet.column_dimensions["D"].width = 18

wb.save(template_path)

print(f"Source Excel: {source_excel}")
print(f"Template saved: {template_path}")
print(f"First sheet inserted: {CONFIG_SHEET_NAME}")

Source Excel: c:\Users\josua\OneDrive\Dokumente\ETH\Code\CS\CS1-Swiss-Post\01-INPUT-DATA\Vétroz\20251023_Standortdaten_Vétroz.xlsx
Template saved: c:\Users\josua\OneDrive\Dokumente\ETH\Code\CS\CS1-Swiss-Post\01-INPUT-DATA\TEMPLATES\template_20251023_Standortdaten_Vétroz_20260320_150043.xlsx
First sheet inserted: CONFIG_INPUTS
